# P9. 이력 관리 테이블(SCD Type 2)

서울시가 반년마다 다시 올리는 따릉이 대여소 목록 네 판(2024년 12월 ~ 2026년 6월 기준)을 비교해 신설 · 폐쇄 · 이름 · 주소 변경을 찾고, valid_from · valid_to · is_current 로 이력을 남기는 SCD Type 2 테이블을 Iceberg 에 병합해 쌓습니다. 「2026년 6월 10일 기준 대여소 목록」을 조회하고, Iceberg 타임트래블과 무엇이 다른지 확인합니다.

- 단계: 개별 프로젝트 · 정형 · 예상 2~3일
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 서울 열린데이터광장 — 공공자전거 대여소 정보(OA-13252) 24.12월 · 25.6월 · 25.12월 · 26.6월 기준 파일(2026-10-29 내려받음), 공공누리 1유형(출처표시)

**빈칸 노트북** — 코드 속 `___①___` 자리 11곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P9/P9_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

따릉이 대여소는 해마다 수십 곳이 새로 생기고, 수십 곳이 문을 닫고, 가끔 이름이나 주소가 바뀝니다.
서울시는 대여소 목록을 반년마다 **새 파일로 통째로** 올립니다. 파일 하나는 「그때의 모습」일 뿐, 무엇이 언제 바뀌었는지는 적혀 있지 않습니다.

마스터 데이터(대여소 · 상품 · 거래처 · 직원처럼 다른 표가 기대는 기준 목록)는 이렇게 「지금 모습」만 오는 경우가 많습니다.
최신 파일로 덮어쓰면 「작년 대여 기록은 그때 어느 대여소 이름이었나」에 답할 수 없습니다. 그래서 바뀔 때마다 옛 줄을 닫고 새 줄을 여는 **SCD Type 2**(Slowly Changing Dimension, 천천히 바뀌는 차원의 2형) 표로 이력을 남깁니다.

```text
대여소 목록 24.12 ─┐
           25.6  ─┤  변경 감지(row_hash) ─ SCD2 병합(upsert) ─ Iceberg 테이블 bike.station_dim
           25.12 ─┤                                          │  station_id · 값들 · valid_from · valid_to · is_current
           26.6  ─┘                                          ├─ 시점 조회: 「2026-06-10 기준 목록」
                                                             └─ 타임트래블: 「세 번째 적재 직후의 테이블」
```

다 끝내면 다음을 말할 수 있습니다.

- 네 판 사이에 실제로 몇 곳이 생기고 · 닫히고 · 바뀌었는지, 그리고 바뀐 내용
- 「2026년 6월 10일에 있던 대여소」를 SCD2 표에서 꺼내는 SQL, 그리고 경계 날짜에서 틀리기 쉬운 곳
- Iceberg 타임트래블(언제 **적재**했나)과 SCD2(현실에서 언제 **유효**했나)가 다른 질문에 답한다는 것
- 같은 변경을 두 번 넣어도 이력이 겹치지 않게 하는 방법(멱등)

## 1단계 · 준비 — 라이브러리와 데이터

Iceberg 는 P1 처럼 **pyiceberg** 로 다룹니다. 카탈로그(테이블 목록)는 sqlite 파일 하나, 데이터는 로컬 폴더에 둡니다(3주차 18장 · 21장의 구성을 서버 없이 줄인 것).
엑셀을 읽는 `openpyxl` 도 확인합니다. Colab 에 없는 것만 설치합니다.

In [ ]:
import sys, os, shutil, subprocess, importlib.util, urllib.request, hashlib, time
from datetime import date, datetime

def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)

need("pyiceberg", "pyiceberg[sql-sqlite,pyarrow]")
need("openpyxl")

import pandas as pd
import pyarrow as pa
import duckdb
import pyiceberg
print("pyiceberg", pyiceberg.__version__, "· duckdb", duckdb.__version__)

데이터는 서울 열린데이터광장 「공공자전거 대여소 정보」(OA-13252)입니다. 이 데이터셋은 반년마다 새 기준의 파일을 더하고, 옛 파일도 내려 두지 않고 남겨 둡니다.
그중 최근 네 판을 받아 두었습니다. 이용허락은 **공공누리 1유형(출처표시)** 이고, 파일은 원본 그대로입니다(이름만 영문으로 바꿈).

| 파일 | 원본 파일 이름 |
|---|---|
| `p9_bike_station_202412.xlsx` | 공공자전거 대여소 정보(24.12월 기준).xlsx |
| `p9_bike_station_202506.xlsx` | 공공자전거 대여소 정보(25.6월 기준).xlsx |
| `p9_bike_station_202512.xlsx` | 공공자전거 대여소 정보(25.12월 기준).xlsx |
| `p9_bike_station_202606.xlsx` | 공공자전거 대여소 정보(26.6월 기준).xlsx |

파일에는 「몇 월 기준」만 있고 날짜는 없습니다. 이 프로젝트에서는 **기준 달의 마지막 날**(2024-12-31 · 2025-06-30 · 2025-12-31 · 2026-06-30)을 그 판의 기준일로 둡니다.

In [ ]:
DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

FILES = {date(2024, 12, 31): "p9_bike_station_202412.xlsx",
         date(2025, 6, 30): "p9_bike_station_202506.xlsx",
         date(2025, 12, 31): "p9_bike_station_202512.xlsx",
         date(2026, 6, 30): "p9_bike_station_202606.xlsx"}
for d, name in FILES.items():
    p = fetch(name)
    print(d, p, round(os.path.getsize(p) / 1e3), "KB")

> **확인 포인트** — 네 파일이 각각 290KB 안팎입니다.

## 2단계 · 네 판 읽기 — 같은 모양으로 맞추기

P2 2단계에서 본 것처럼 이 엑셀은 위 다섯 줄이 병합된 제목입니다. 먼저 그대로 읽으면 어떻게 되는지 봅니다.

In [ ]:
naive = pd.read_excel("data/p9_bike_station_202606.xlsx", dtype=str)
print(naive.columns.tolist()[:6])
try:
    pd.to_numeric(naive.iloc[:, 4])                  # 위도 열이라고 생각한 곳
except ValueError as e:
    print("숫자로 못 바꿈:", str(e)[:60])

제목 줄을 건너뛰고, 쓸 열에 이름을 붙이는 함수를 만듭니다. 이력으로 남길 값은 여섯 가지입니다 — **이름 · 자치구 · 주소 · 위도 · 경도 · 거치대 수**.
거치대 수는 파일에 LCD 거치대와 QR 거치대가 따로 적혀 있어 둘을 더합니다(한쪽만 있는 대여소가 많아 빈칸은 0으로 봅니다).

In [ ]:
TRACK = ["name", "gu", "addr", "lat", "lon", "docks"]

def read_station(path):
    r = pd.read_excel(path, header=None, skiprows=___①___, dtype=str)   # ① 제목 줄 수
    r.columns = ["station_id", "name", "gu", "addr", "lat", "lon", "installed", "lcd", "qr", "op"]
    for c in ["name", "gu", "addr"]:
        r[c] = r[c].str.strip()
    r["lat"] = pd.to_numeric(r["lat"])
    r["lon"] = pd.to_numeric(r["lon"])
    r["docks"] = (pd.to_numeric(r["lcd"]).fillna(0) + pd.to_numeric(r["qr"]).___②___(0)).astype("int32")   # ② 빈칸을 0으로
    r["installed"] = pd.to_datetime(r["installed"], errors="coerce").dt.date
    return r[["station_id"] + TRACK + ["installed", "lcd", "qr"]]

snaps = {d: read_station(os.path.join("data", f)) for d, f in FILES.items()}
for d, s in snaps.items():
    print(d, len(s), "곳 · 번호 겹침 없음:", s["station_id"].is_unique, "· 거치대 합", s["docks"].sum())
snaps[date(2026, 6, 30)].head(3)

> **확인 포인트** — 2,766곳 → 2,780곳 → 2,799곳 → **2,789곳**. 네 판 모두 대여소 번호가 겹치지 않습니다 — 번호를 이력의 키로 씁니다.

In [ ]:
assert [len(s) for s in snaps.values()] == [2766, 2780, 2799, 2789]
assert all(s["station_id"].is_unique for s in snaps.values())

## 3단계 · 변경 감지 — 무엇이 바뀌었나

두 판을 대여소 번호로 맞대어 비교합니다. 먼저 일부러 단순하게, 열마다 `!=` 로 비교해 봅니다.

In [ ]:
old, new = snaps[date(2025, 12, 31)].set_index("station_id"), snaps[date(2026, 6, 30)].set_index("station_id")
both = old.index.intersection(new.index)
naive_diff = (old.loc[both, ["name", "lcd"]] != new.loc[both, ["name", "lcd"]]).sum()
print(naive_diff.to_dict())

LCD 거치대가 1,700곳 넘게 「바뀌었다」고 나옵니다. 사실은 두 판 모두 비어 있는 칸입니다 — `NaN != NaN` 은 언제나 참이라 빈칸끼리 비교하면 다르다고 나옵니다.
그래서 비교할 값을 **글자 하나로 이어 붙여 해시(row_hash)** 를 만들고, 해시만 비교합니다. 빈칸도 같은 글자(`nan`)가 되어 같다고 나오고, 열이 몇 개든 비교는 한 번입니다.

In [ ]:
def add_hash(df):
    joined = df[TRACK].astype(str).agg("|".join, axis=1)
    df["row_hash"] = joined.map(lambda s: hashlib.___③___(s.encode()).hexdigest())   # ③ 해시 함수
    return df

for s in snaps.values():
    add_hash(s)

def diff(a, b):
    """두 판(a → b)의 신설 · 폐쇄 · 변경 대여소 번호."""
    m = a[["station_id", "row_hash"]].merge(b[["station_id", "row_hash"]], on="station_id",
                                            how="outer", suffixes=("_a", "_b"), indicator=True)
    return {"new": set(m.loc[m["_merge"] == "___④___", "station_id"]),   # ④ 새 판에만 있는
            "gone": set(m.loc[m["_merge"] == "left_only", "station_id"]),
            "changed": set(m.loc[(m["_merge"] == "both") & (m["row_hash_a"] != m["row_hash_b"]), "station_id"])}

days = list(FILES)
for a, b in zip(days, days[1:]):
    d = diff(snaps[a], snaps[b])
    print(a, "→", b, {k: len(v) for k, v in d.items()})

> **확인 포인트** — 반년마다 신설 27 · 43 · 9곳, 폐쇄 13 · 24 · 19곳이고, 값이 바뀐 대여소는 마지막 판에서만 **3곳**입니다.

In [ ]:
counts = [{k: len(v) for k, v in diff(snaps[a], snaps[b]).items()} for a, b in zip(days, days[1:])]
assert counts == [{"new": 27, "gone": 13, "changed": 0}, {"new": 43, "gone": 24, "changed": 0}, {"new": 9, "gone": 19, "changed": 3}]

바뀐 3곳을 나란히 봅니다.

In [ ]:
chg = sorted(diff(snaps[days[2]], snaps[days[3]])["changed"])
pd.concat({"25.12": old.loc[chg, ["name", "addr"]], "26.6": new.loc[chg, ["name", "addr"]]}, axis=1)

> **확인 포인트** — 5896 `영드포구청역 6번출구` → `영등포구청역 6번출구`(오타 정정), 3939 `구로구새말로` → `구로구 새말로`(띄어쓰기), 4080 은 주소 칸에 들어가 있던 시민 의견 문장이 지워졌습니다.
> 실제 자료에서는 거치대 수 · 좌표가 바뀐 대여소가 네 판 동안 한 곳도 없었습니다. 큰 변화는 신설 · 폐쇄이고, 값 변경은 대부분 **정정**입니다.

In [ ]:
assert chg == ["3939", "4080", "5896"]
assert new.loc["5896", "name"] == "영등포구청역 6번출구"

> **Aebon's Tip** — 오타 정정까지 새 이력으로 남길지는 정해야 할 문제입니다. 「그때 화면에 그렇게 보였다」를 재현해야 하면 남기고, 분석만 할 거라면 정정은 옛 줄을 고쳐 쓰는 SCD **Type 1** 로 처리하기도 합니다. 여기서는 모두 Type 2 로 남깁니다.

## 4단계 · 첫 적재 — SCD2 테이블 만들기

SCD2 표에는 값 열 옆에 세 열이 붙습니다.

| 열 | 뜻 |
|---|---|
| `valid_from` | 이 줄의 값이 맞기 시작한 날(포함) |
| `valid_to` | 이 줄의 값이 끝난 날(포함 안 함). 아직 유효하면 `9999-12-31` |
| `is_current` | 지금 유효한 줄인가 — 「최신 목록」을 빨리 꺼내려고 둡니다 |

**valid_from 을 무엇으로 잡을지**가 첫 고민입니다. 신설 대여소는 파일의 `설치 시기`가 실제 시작일이라 그것을 씁니다(앞 기준일과 이번 기준일 사이일 때만).
폐쇄 · 변경은 언제 일어났는지 파일에 없으니, **그 변화를 처음 본 판의 기준일**로 잡습니다 — 「늦어도 이날까지는 바뀌었다」는 뜻입니다.
첫 판에 있던 대여소는 설치 시기부터 지금 값이 유효했다고 둡니다(그 사이의 변화는 알 수 없다는 가정입니다).

In [ ]:
END = date(9999, 12, 31)
COLS = ["station_id"] + TRACK + ["row_hash", "valid_from", "valid_to", "is_current"]

def open_versions(df, as_of, prev_as_of=None, new_ids=None):
    """목록의 줄들을 「지금부터 유효한」 SCD2 줄로 바꿉니다."""
    out = df.copy()
    def start(sid, inst):
        brand_new = new_ids is None or sid in new_ids
        ok = pd.notna(inst) and inst <= as_of and (prev_as_of is None or inst > prev_as_of)
        return inst if (brand_new and ok) else as_of
    out["valid_from"] = [start(s, i) for s, i in zip(out["station_id"], out["installed"])]
    out["valid_to"] = ___⑤___   # ⑤ 아직 끝나지 않음
    out["is_current"] = True
    return out[COLS]

first = open_versions(snaps[days[0]], days[0])
print(first["valid_from"].min(), "~", first["valid_from"].max())
first.head(3)

pyiceberg 카탈로그를 만들고, 첫 판을 테이블에 넣습니다. 다시 실행해도 되도록 같은 이름의 테이블이 있으면 지우고 만듭니다.
적재 기록(`bike.load_log`)도 함께 만듭니다 — 어느 기준일 판을 언제 넣었는지 남겨 8단계의 멱등 처리에 씁니다.

In [ ]:
from pyiceberg.catalog.sql import SqlCatalog

WAREHOUSE = os.path.abspath("warehouse")
os.makedirs(WAREHOUSE, exist_ok=True)
catalog = SqlCatalog("lake", uri=f"sqlite:///{WAREHOUSE}/catalog.db", warehouse=f"file://{WAREHOUSE}")
catalog.create_namespace_if_not_exists("bike")
for t in ["bike.station_dim", "bike.load_log"]:
    if catalog.table_exists(t):
        catalog.drop_table(t)

first_arrow = pa.Table.from_pandas(first, preserve_index=False)
dim = catalog.create_table("bike.station_dim", schema=first_arrow.schema)
dim.___⑥___(first_arrow)   # ⑥ 행 덧붙이기

LOG = pa.schema([("as_of", pa.date32()), ("src", pa.string()), ("new", pa.int32()), ("gone", pa.int32()),
                 ("changed", pa.int32()), ("loaded_ms", pa.int64())])
log = catalog.create_table("bike.load_log", schema=LOG)

def write_log(as_of, src, n_new, n_gone, n_changed):
    log.append(pa.Table.from_pylist([{"as_of": as_of, "src": src, "new": n_new, "gone": n_gone, "changed": n_changed,
                                      "loaded_ms": int(time.time() * 1000)}], schema=LOG))

write_log(days[0], FILES[days[0]], len(first), 0, 0)
print("행:", len(dim.scan().to_arrow()), "· 스냅샷:", len(dim.snapshots()))

> **확인 포인트** — 행 **2,766**, 스냅샷 1개. 모두 `is_current = True` 입니다.

In [ ]:
cur0 = dim.scan().to_pandas()
assert len(cur0) == 2766 and cur0["is_current"].all() and len(dim.snapshots()) == 1

형이 다른 표를 넣으면 Iceberg 는 테이블 스키마와 맞지 않는다며 거절합니다. 거치대 수를 글자로 바꾼 두 줄을 넣어 봅니다.

In [ ]:
try:
    dim.append(pa.Table.from_pandas(first.head(2).astype({"docks": str}), preserve_index=False))
except ValueError as e:
    print("적재 거절:", str(e).splitlines()[0][:100])
print("행 수 그대로:", len(dim.scan().to_arrow()))
assert len(dim.scan().to_arrow()) == 2766

> **자주 겪는 오류** — 반대로 **열이 빠진** 표는 거절되지 않습니다. pyarrow 에서 만든 열은 모두 빈칸을 허용(optional)하므로, 빠진 열은 빈칸으로 채워져 들어갑니다.
> `is_current` 가 빈 줄이 섞이면 「현재 목록」에서 조용히 빠집니다. 꼭 있어야 하는 열은 스키마를 만들 때 `required=True` 로 두거나, 넣기 전에 열을 확인하세요(P7 의 R01 규칙).

## 5단계 · SCD2 병합 — 닫고, 열고

새 판이 오면 할 일은 셋입니다.

1. 테이블의 **현재 줄**(`is_current`)과 새 판을 해시로 비교합니다.
2. 사라진 대여소 · 값이 바뀐 대여소의 현재 줄을 **닫습니다** — `valid_to` = 이번 기준일, `is_current` = False.
3. 새로 생긴 대여소 · 값이 바뀐 대여소의 새 줄을 **엽니다** — `valid_from` 부터 `9999-12-31` 까지.

닫기는 이미 있는 줄을 고치는 일(update)이고, 열기는 새 줄을 넣는 일(insert)입니다. pyiceberg 의 `upsert` 는 둘을 한 번에 합니다 — 키 열이 같은 줄이 있으면 고치고, 없으면 넣습니다.
키는 **(station_id, valid_from)** 입니다. 대여소 번호만으로는 한 대여소의 여러 이력 줄을 가를 수 없기 때문입니다.

In [ ]:
def merge_snapshot(df, as_of):
    """새 판 하나를 SCD2 테이블에 병합합니다. 바뀐 개수를 돌려줍니다."""
    cur = dim.scan(row_filter="is_current == true").to_pandas()
    prev_as_of = max(r["as_of"] for r in log.scan().to_arrow().to_pylist())
    d = diff(cur, df)                                         # 이번에는 「테이블의 현재 줄」과 비교
    closing = cur[cur["station_id"].isin(d["gone"] | d["changed"])].assign(valid_to=as_of, is_current=___⑦___)   # ⑦ 더는 현재가 아님
    opening = open_versions(df[df["station_id"].isin(d["new"] | d["changed"])], as_of, prev_as_of, d["new"])
    rows = pd.concat([closing, opening], ignore_index=True)[COLS]
    if len(rows):
        res = dim.___⑧___(pa.Table.from_pandas(rows, preserve_index=False, schema=dim.schema().as_arrow()),   # ⑧ 있으면 고치고 없으면 넣기
                         join_cols=["station_id", "valid_from"])
        print(as_of, f"닫음 {res.rows_updated} · 엶 {res.rows_inserted}")
    return {k: len(v) for k, v in d.items()}

for as_of in days[1:]:
    n = merge_snapshot(snaps[as_of], as_of)
    write_log(as_of, FILES[as_of], n["new"], n["gone"], n["changed"])
    time.sleep(1)                                             # 스냅샷 시각이 확실히 갈리게(7단계)

hist = dim.scan().to_pandas()
print("전체 줄:", len(hist), "· 현재 줄:", hist["is_current"].sum())

> **확인 포인트** — 25.6 판 닫음 13 · 엶 27, 25.12 판 닫음 24 · 엶 43, 26.6 판 닫음 22(폐쇄 19 + 변경 3) · 엶 12(신설 9 + 변경 3).
> 전체 줄 **2,848**, 현재 줄 **2,789** — 26.6 판의 대여소 수와 같습니다.

In [ ]:
assert len(hist) == 2848 and hist["is_current"].sum() == 2789
assert set(hist.loc[hist["is_current"], "station_id"]) == set(snaps[days[3]]["station_id"])
assert hist.groupby("station_id")["is_current"].sum().max() == 1          # 대여소마다 현재 줄은 하나

이름이 바뀐 5896 의 이력입니다. 옛 줄은 2026-06-30 에 닫히고 같은 날 새 줄이 열렸습니다.

In [ ]:
hist[hist["station_id"] == "5896"].sort_values("valid_from")[["station_id", "name", "valid_from", "valid_to", "is_current"]]

In [ ]:
h = hist[hist["station_id"] == "5896"].sort_values("valid_from")
assert h["name"].tolist() == ["영드포구청역 6번출구", "영등포구청역 6번출구"]
assert h["valid_to"].iloc[0] == h["valid_from"].iloc[1] == date(2026, 6, 30)

`upsert` 는 넣는 표 안에 같은 키가 두 번 있으면 어느 쪽을 써야 할지 몰라 거절합니다. 병합할 줄을 만들다 중복을 섞는 실수를 여기서 막아 줍니다.

In [ ]:
one = open_versions(snaps[days[3]].head(1), days[3])
dup = pd.concat([one, one])
try:
    dim.upsert(pa.Table.from_pandas(dup, preserve_index=False, schema=dim.schema().as_arrow()), join_cols=["station_id", "valid_from"])
except Exception as e:
    print(type(e).__name__, "—", str(e).splitlines()[0][:90])

## 6단계 · 시점 조회 — 「2026년 6월 10일 기준 대여소 목록」

SCD2 표에서 어느 날짜 `D` 의 목록은 「그날 유효했던 줄」입니다.

```text
valid_from <= D  AND  D < valid_to
```

시작은 포함하고 끝은 포함하지 않는(반열린 구간) 규칙 덕분에, 한 대여소의 줄들이 겹치지도 비지도 않고 이어집니다.

In [ ]:
dim_arrow = dim.scan().to_arrow()

def as_of(day):
    return duckdb.sql(f"""
        SELECT station_id, name, gu, docks FROM dim_arrow
        WHERE valid_from <= DATE '{day}' AND DATE '{day}' ___⑨___ valid_to   -- ⑨ 끝 날짜는 포함하지 않음
    """).df()

jun10 = as_of("2026-06-10")
print("2026-06-10:", len(jun10), "곳 · 2025-06-10:", len(as_of("2025-06-10")), "곳")
jun10.groupby("gu").size().sort_values(ascending=False).head(5)

> **확인 포인트** — 2026-06-10 기준 **2,808곳**, 2025-06-10 기준 **2,787곳**입니다.
> 2,808 = 25.12 판 2,799곳 + 26.6 판에 새로 생긴 9곳(설치 시기가 모두 6월 10일 전). 26.6 판에서 사라진 19곳은 「6월 30일까지는 닫혔다」는 것만 알므로 6월 10일 목록에는 아직 들어 있습니다.

In [ ]:
assert len(jun10) == 2808 and jun10["station_id"].is_unique
assert len(as_of("2025-06-10")) == 2787

이번에는 일부러 `BETWEEN`(양 끝 포함)으로 써 봅니다. 평소에는 같은 답이 나오지만, **기준일 당일**에는 닫힌 줄과 열린 줄이 둘 다 걸립니다.

In [ ]:
def as_of_between(day):
    return duckdb.sql(f"SELECT station_id FROM dim_arrow WHERE DATE '{day}' BETWEEN valid_from AND valid_to").df()

for day in ["2026-06-10", "2026-06-30"]:
    a, b = as_of(day), as_of_between(day)
    print(day, "반열린:", len(a), "· BETWEEN:", len(b), "· 번호 겹침:", b["station_id"].duplicated().sum())

> **확인 포인트** — 6월 10일은 둘 다 2,808곳이지만, 6월 30일은 반열린 2,789곳 · BETWEEN 2,811곳(번호 겹침 3)입니다 — 그날 닫힌 19곳과, 이름이 바뀐 3곳의 옛 줄이 함께 걸립니다.

In [ ]:
assert len(as_of_between("2026-06-30")) == 2811 and as_of_between("2026-06-30")["station_id"].duplicated().sum() == 3
assert len(as_of("2026-06-30")) == 2789

> **자주 겪는 오류** — `valid_to` 를 「마지막으로 유효한 날」(닫힌 구간)로 쓰는 팀도 있습니다. 그때는 `valid_to = 기준일 - 1일` 로 닫고 BETWEEN 을 씁니다. 어느 쪽이든 괜찮지만 **한 표 안에서는 하나로만** 정하세요.

> **Aebon's Tip** — 따릉이 대여 기록(M1 · P2)과 이을 때도 같은 조건을 씁니다. `JOIN station_dim s ON r.station_id = s.station_id AND s.valid_from <= r.rent_date AND r.rent_date < s.valid_to` — 「빌린 날 그 대여소의 이름 · 자치구」가 붙습니다.

## 7단계 · 타임트래블과 SCD2 — 무엇이 다른가

Iceberg 는 적재할 때마다 스냅샷을 남기므로(P1 7단계), 지난 스냅샷을 열면 「그때의 테이블」이 나옵니다. 그러면 SCD2 가 왜 필요할까요?

In [ ]:
snap = dim.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "operation"]]
print(len(snap), "개 스냅샷")
snap

스냅샷 시각은 **이 노트북을 실행한 시각**입니다. 2026년 6월 10일에 이 테이블은 아직 없었으니, 그 시각으로 찾으면 스냅샷이 없습니다.

In [ ]:
t = int(datetime(2026, 6, 10).timestamp() * 1000)
found = dim.___⑩___(t)   # ⑩ 시각으로 스냅샷 찾기
print("2026-06-10 의 스냅샷:", found)
try:
    dim.scan(snapshot_id=found.snapshot_id).to_arrow()
except AttributeError as e:
    print("타임트래블 실패:", e)

타임트래블로 할 수 있는 것은 「**세 번째 적재(25.12 판)가 끝난 직후**의 테이블」을 여는 일입니다. 적재 기록의 시각으로 찾습니다.

In [ ]:
logs = log.scan().to_pandas().sort_values("as_of")
t3 = int(logs.iloc[2]["loaded_ms"]) + 500          # 25.12 판을 넣고 0.5초 뒤(적재 기록은 병합이 끝난 뒤 남깁니다)
after3 = dim.scan(snapshot_id=dim.snapshot_as_of_timestamp(t3).snapshot_id).to_pandas()
print("세 번째 적재 직후 — 전체 줄:", len(after3), "· 현재 줄:", after3["is_current"].sum())
print("SCD2 로 본 2025-12-31 목록:", len(as_of("2025-12-31")))

> **확인 포인트** — 세 번째 적재 직후 전체 2,836줄 · 현재 2,799줄이고, SCD2 로 2025-12-31 기준을 물어도 2,799곳 — 이 테이블에서는 두 답이 같습니다.
> 같은 것은 판을 **기준일 순서대로, 빠짐없이** 넣었기 때문입니다. 늦게 받은 판, 나중에 고친 판이 섞이면 둘은 갈라집니다.

In [ ]:
assert len(after3) == 2836 and after3["is_current"].sum() == 2799
assert len(as_of("2025-12-31")) == 2799

두 방식은 서로 다른 시간에 답합니다.

| | Iceberg 타임트래블 | SCD Type 2 |
|---|---|---|
| 시간의 뜻 | **적재 시간** — 테이블에 언제 쓰였나 | **유효 시간** — 현실에서 언제 그 값이었나 |
| 답하는 질문 | 「어제 회의 때 대시보드가 본 테이블은?」 | 「2026년 6월 10일에 있던 대여소는?」 |
| 시간 축을 만드는 것 | 엔진(커밋할 때 저절로) | 데이터(우리가 valid_from 을 정함) |
| 얼마나 남나 | 스냅샷을 정리(expire)하면 사라짐 | 줄로 남아 지우지 않는 한 영구 |
| 늦게 온 자료 · 정정 | 받은 날의 스냅샷으로 기록 | 과거 날짜로 이력을 끼워 넣을 수 있음 |

둘은 함께 씁니다. SCD2 테이블을 Iceberg 에 두면 「2026년 6월 10일 기준 목록을, **지난주에 우리가 알던 대로** 보여 줘」(유효 시간 × 적재 시간, 이른바 이중 시간 bitemporal)까지 답할 수 있습니다.

## 8단계 · 멱등 처리 — 같은 변경을 두 번 넣으면

배치 작업은 실패하면 다시 돌립니다(8주차 Airflow 의 재시도). 그런데 변경을 **사건(이벤트)** 으로 받아 그대로 덧붙이는 방식이면, 같은 묶음이 두 번 들어올 때 이력이 겹칩니다.
26.6 판을 넣기 직전의 테이블(타임트래블로 꺼냅니다)에, 「25.12 → 26.6 변경 묶음」을 두 번 적용해 봅니다. 이 실험은 pandas 표에서만 합니다(Iceberg 테이블은 건드리지 않음).

In [ ]:
events = []
d = diff(snaps[days[2]], snaps[days[3]])                         # 파일끼리 비교해 만든 변경 묶음
new_df = snaps[days[3]].set_index("station_id")
for sid in sorted(d["new"]):
    events.append(("new", sid))
for sid in sorted(d["changed"]):
    events.append(("change", sid))
for sid in sorted(d["gone"]):
    events.append(("close", sid))

def apply_events(table, events, as_of):
    """변경 사건을 차례로 덧붙입니다 — 테이블의 지금 상태를 보지 않습니다."""
    t = table.copy()
    add = []
    for op, sid in events:
        if op in ("change", "close"):
            m = (t["station_id"] == sid) & t["is_current"]
            t.loc[m, ["valid_to", "is_current"]] = [as_of, False]
        if op in ("new", "change"):
            add.append(open_versions(new_df.loc[[sid]].reset_index(), as_of, days[2], d["new"]))
    return pd.concat([t] + add, ignore_index=True)

before = dim.scan(snapshot_id=dim.snapshot_as_of_timestamp(t3).snapshot_id).to_pandas()
once = apply_events(before, events, days[3])
twice = apply_events(once, events, days[3])                      # 재시도로 같은 묶음이 한 번 더
for name, t in [("한 번", once), ("두 번", twice)]:
    cur_per = t[t["is_current"]].groupby("station_id").size()
    zero = (t["valid_from"] == t["valid_to"]).sum()
    print(name, "— 줄:", len(t), "· 현재 줄이 둘 이상인 대여소:", (cur_per > 1).sum(), "· 길이 0인 이력:", zero)

> **확인 포인트** — 한 번 넣으면 2,848줄로 문제가 없지만, 두 번 넣으면 **2,860줄** — 신설 9곳은 현재 줄이 둘씩 생기고, 변경 3곳에는 `valid_from = valid_to` 인 길이 0짜리 이력이 끼어듭니다.

In [ ]:
assert len(once) == 2848 and len(twice) == 2860
assert (twice[twice["is_current"]].groupby("station_id").size() > 1).sum() == 9
assert (twice["valid_from"] == twice["valid_to"]).sum() == 3

5단계의 `merge_snapshot` 은 두 번 돌려도 안전합니다. 사건이 아니라 **테이블의 현재 상태와 새 판을 비교**하고, 키 (station_id, valid_from) 로 upsert 하기 때문입니다. 같은 26.6 판을 한 번 더 병합해 봅니다.

In [ ]:
n_before = len(dim.scan().to_arrow())
again = merge_snapshot(snaps[days[3]], days[3])
print("다시 병합:", again, "· 줄 수", n_before, "→", len(dim.scan().to_arrow()))
assert again == {"new": 0, "gone": 0, "changed": 0} and len(dim.scan().to_arrow()) == 2848

한 겹 더 막습니다. 적재 기록을 보고 **이미 넣은 기준일이거나 그보다 옛 판이면** 아예 시작하지 않습니다. 옛 판이 늦게 들어와 최신 상태를 거꾸로 되돌리는 사고까지 막아 줍니다.

In [ ]:
def load_file(as_of):
    done = [r["as_of"] for r in log.scan().to_arrow().to_pylist()]
    if as_of ___⑪___ max(done):   # ⑪ 이미 넣었거나 더 옛날
        return f"건너뜀 — {as_of} 판은 이미 넣었거나 최신({max(done)})보다 옛 판입니다"
    n = merge_snapshot(snaps[as_of], as_of)
    write_log(as_of, FILES[as_of], n["new"], n["gone"], n["changed"])
    return n

print(load_file(date(2026, 6, 30)))
print(load_file(date(2025, 6, 30)))
assert len(dim.scan().to_arrow()) == 2848 and len(log.scan().to_arrow()) == 4

> **자주 겪는 오류** — 옛 판을 「건너뛰기」가 늘 정답은 아닙니다. 놓친 판을 나중에 받았다면 그 사이 이력을 끼워 넣어야 합니다(늦게 온 자료, late-arriving data).
> 그때는 거꾸로 되돌리지 말고, 그 판의 기준일로 줄을 나눠 넣는 별도의 「되채우기(backfill)」 작업을 만드세요.

## 9단계 · 더 해 보기

- OA-13252 에는 21.01 부터 반년마다의 판이 남아 있습니다. 데이터셋 화면에서 옛 판(22.12 · 23.6 · 23.12 · 24.6 등)을 더 받아 순서대로 병합하고, 몇 년 동안 이름이 가장 많이 바뀐 대여소를 찾아 보세요(파일마다 열 구성이 같은지부터 확인).
- 오타 정정은 Type 1(덮어쓰기), 이전 · 이름 변경은 Type 2 로 나누려면 `TRACK` 을 두 묶음으로 나누고 해시를 둘 만들면 됩니다. 어떤 열을 어느 쪽에 둘지 정해 보세요.
- M1 의 대여 기록(2026-06-05~07)을 6단계 Tip 의 조건으로 `station_dim` 과 이어, 대여소 이름이 그날 기준으로 붙는지 확인하세요. 번호가 이력에 없는 대여소는 몇 곳인가요?
- Iceberg 의 `dim.maintenance.expire_snapshots()`(또는 Spark 의 `CALL system.expire_snapshots`)로 오래된 스냅샷을 정리한 뒤 7단계를 다시 해 보세요. 타임트래블은 사라져도 SCD2 이력은 그대로 남습니다.

## 정답

① `5` — 제목 줄 수
② `fillna` — 빈칸을 0으로
③ `md5` — 해시 함수
④ `right_only` — 새 판에만 있는
⑤ `END` — 아직 끝나지 않음
⑥ `append` — 행 덧붙이기
⑦ `False` — 더는 현재가 아님
⑧ `upsert` — 있으면 고치고 없으면 넣기
⑨ `<` — 끝 날짜는 포함하지 않음
⑩ `snapshot_as_of_timestamp` — 시각으로 스냅샷 찾기
⑪ `<=` — 이미 넣었거나 더 옛날